# ***Assignment 2:*** Paper Reproduction

## **Members:** *Muhammad Huzaifa Khalid (23i2508), Waqas Ahmed (23i-2540), Abubakar Imran (23i-2589)*

## **Paper:** *"Enhanced credit risk prediction using deep learning and SMOTE-ENN resampling"* by Idowu Aruleba and Yanxia Sun.*

## **Objective:**
This notebook implements a controlled, end-to-end reproduction of the core claims of the selected research paper. Our implementation rigorously tests three models: **MLP, LSTM, and GRU** across both the **German and Australian** credit risk datasets, under two major experimental conditions: **Baseline (no resampling)** and **SMOTE-ENN resampling**.

## **Reproducibility Analysis & Core Scope:**
This notebook serves as both an implementation and a reproducibility audit. Important discrepancies found during our reproduction attempt—such as contradictions in the original paper's datasets descriptions and omitted structural variables—are safely accommodated and carefully documented. Models that were under-specified in the original text (such as CNN/GNN architectures lacking spatial or arbitrary structural definitions) are excluded in favor of rigorously defined temporal and fully connected states.

---

***This Block Is Exclusivly For Testing***

TESTING MODE = Set to True for quick validation, False for full reproduction

In [1]:
TESTING_MODE = False

# Global flags
if TESTING_MODE:
    NUM_EPOCHS = 20  # Test with 20 epochs
    print("TESTING MODE ACTIVE: Using 20 epochs instead of 100")
else:
    NUM_EPOCHS = 100  # Full reproduction
    print("FULL MODE: Using 100 epochs as per paper")


FULL MODE: Using 100 epochs as per paper


## ***Environment, Hardware, and Imports Setup***
***Purpose:*** Ensure all standard scientific computing and deep learning dependencies are fully available, imported safely, and configure optimal runtime hardware parameters.

***Hardware Scaling Strategy:*** 
The notebook script relies heavily on `torch.cuda`. It operates dynamically across local topologies or cloud (e.g., Kaggle free-tier `T4 x2`). Data parallelism is enforced strictly when `torch.cuda.device_count() > 1`. Mixed precision acceleration (`torch.cuda.amp.autocast`) minimizes VRAM consumption and accelerates throughput smoothly.

***Mixed Precision Training:***
We are using PyTorch's Automatic Mixed Precision (AMP) to:
- Reduce VRAM consumption efficiently.
- Accelerate training significantly on capable GPUs.
- Maintain numerical stability for credit risk prediction constraints.

***Reproducibility Note:*** AMP may introduce minor (<0.1%) numerical variations across different GPU architectures due to different hardware FP16 implementations.


***Importing All The Necessary Libraries***

In [2]:
import os
import sys
import random
import logging
import time
import datetime
import warnings
import platform
import gc
import copy
from pathlib import Path
import json
import shutil

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset, random_split

import sklearn
from sklearn.preprocessing import StandardScaler, OrdinalEncoder
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score, confusion_matrix, recall_score

import imblearn
from imblearn.combine import SMOTEENN

import shap

In [3]:
warnings.filterwarnings('ignore')

***Directory Scaffold***

In [4]:
dirs_required = [
    'results/logs',
    'results/metrics',
    'results/histories',
    'results/tables',
    'results/plots',
    'results/shap'
]
for d in dirs_required:
    os.makedirs(d, exist_ok=True)

***Centralized Logging***

In [5]:
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s',
    handlers=[logging.StreamHandler(), logging.FileHandler('results/logs/master_execution.log')]
)
logger = logging.getLogger('ReproducibilityLog')

***Dynamic Node Extraction***

In [6]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
device_count = torch.cuda.device_count()

num_workers = 2 if torch.cuda.is_available() else 0
pin_memory = True if torch.cuda.is_available() else False

logger.info(f"Target Device: {device} | Discovered GPUs: {device_count}")

2026-04-05 05:15:19,945 - INFO - Target Device: cuda | Discovered GPUs: 2


## ***Environment Print & Hardware Registration***
***Purpose:*** Produce an explicit log registering local runtime context variables requested within the formal reproducibility constraints (Python versions, PyTorch variants, OS).


In [7]:
def PrintEnvironmentSummary():
    env_info = {
        "Python": sys.version,
        "System": platform.platform(),
        "PyTorch": torch.__version__,
        "NumPy": np.__version__,
        "Pandas": pd.__version__,
        "Scikit-Learn": sklearn.__version__,
        "Imbalanced-Learn": imblearn.__version__,
        "SHAP": shap.__version__,
        "CUDA Available": torch.cuda.is_available(),
        "GPU Count": device_count,
        "GPU Name(s)": [torch.cuda.get_device_name(i) for i in range(device_count)] if device_count > 0 else ["N/A"]
    }
    
    logger.info("=== Environment Hardware & Package Summary ===")
    with open("results/logs/environment_summary.json", "w") as f:
        import json
        json.dump(env_info, f, indent=4)
        
    for k, v in env_info.items():
        print(f"{k}: {v}")
        
PrintEnvironmentSummary()

2026-04-05 05:15:19,970 - INFO - === Environment Hardware & Package Summary ===


Python: 3.12.12 (main, Oct 10 2025, 08:52:57) [GCC 11.4.0]
System: Linux-6.6.113+-x86_64-with-glibc2.35
PyTorch: 2.10.0+cu128
NumPy: 2.0.2
Pandas: 2.3.3
Scikit-Learn: 1.6.1
Imbalanced-Learn: 0.14.1
SHAP: 0.50.0
CUDA Available: True
GPU Count: 2
GPU Name(s): ['Tesla T4', 'Tesla T4']


## ***Reproducibility Bounds & Seed Control***
***Purpose:*** Hard-lock environments globally.


In [8]:
def SetSeed(seed_value=42):
    random.seed(seed_value)
    np.random.seed(seed_value)
    os.environ['PYTHONHASHSEED'] = str(seed_value)
    torch.manual_seed(seed_value)
    torch.cuda.manual_seed(seed_value)
    torch.cuda.manual_seed_all(seed_value)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    logger.info(f"RNG components uniformly saturated against fixed seed: {seed_value}")

SetSeed(42)

2026-04-05 05:15:19,994 - INFO - RNG components uniformly saturated against fixed seed: 42


## ***Dataset Validation and Extract Sequences***
***Purpose:*** This code will actively assert path configurations ensuring nodes fall back gracefully or explicitly declare boundaries when running Kaggle native configurations vs Local extraction protocols.


***Dataset Path Validation***

In [9]:
german_path_kaggle = "/kaggle/input/datasets/huzaifakhalid7/dataset/australian.dat"
australian_path_kaggle = "/kaggle/input/datasets/huzaifakhalid7/dataset/australian.dat"

In [10]:
german_target_path = german_path_kaggle if os.path.exists(german_path_kaggle) else 'dataset/german.data-numeric'
if not os.path.exists(german_target_path):
    german_target_path = '../dataset/german.data-numeric'

australian_target_path = australian_path_kaggle if os.path.exists(australian_path_kaggle) else 'dataset/australian.dat'
if not os.path.exists(australian_target_path):
    australian_target_path = '../dataset/australian.dat'

logger.info("=== Dataset Validation ===")
# Check German dataset
if not os.path.exists(german_target_path):
    error_msg = f"German dataset not found at {german_target_path}"
    logger.error(error_msg)
    logger.error("Please verify Kaggle dataset is properly attached or localized:")
    raise FileNotFoundError(error_msg)
else:
    logger.info(f"✓ German dataset found: {german_target_path}")

# Check Australian dataset
if not os.path.exists(australian_target_path):
    error_msg = f"Australian dataset not found at {australian_target_path}"
    logger.error(error_msg)
    raise FileNotFoundError(error_msg)
else:
    logger.info(f"✓ Australian dataset found: {australian_target_path}")
logger.info("All datasets validated successfully")

def LoadGermanDataset(filepath=german_target_path):
    german_frame = pd.read_csv(filepath, sep=r'\s+', header=None)
    return german_frame

def LoadAustralianDataset(filepath=australian_target_path):
    australian_frame = pd.read_csv(filepath, sep=r'\s+', header=None, na_values=['?'])
    return australian_frame

df_german = LoadGermanDataset()
df_australian = LoadAustralianDataset()


2026-04-05 05:15:20,017 - INFO - === Dataset Validation ===
2026-04-05 05:15:20,018 - INFO - ✓ German dataset found: /kaggle/input/datasets/huzaifakhalid7/dataset/australian.dat
2026-04-05 05:15:20,019 - INFO - ✓ Australian dataset found: /kaggle/input/datasets/huzaifakhalid7/dataset/australian.dat
2026-04-05 05:15:20,019 - INFO - All datasets validated successfully


## ***Subsets Assessment & Initial Dimensional Integrity Validate***
***Purpose:*** Display local layout parameters explicitly referencing target configurations.


***German Dataset Paramters***

In [11]:
print("----- German Dataset Parameters -----")
print(f"Dimension Layout (Rows x Cols): {df_german.shape}")
print(f"Distribution Vector (Col 24): {df_german.iloc[:, -1].value_counts().to_dict()}")
print(f"Aggregated Missing Anomalies: {df_german.isna().sum().sum()}")
display(df_german.head(2))

----- German Dataset Parameters -----
Dimension Layout (Rows x Cols): (690, 15)
Distribution Vector (Col 24): {0: 383, 1: 307}
Aggregated Missing Anomalies: 0


,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14
0,1,22.08,11.46,2,4,4,1.585,0,0,0,1,2,100,1213,0
1,0,22.67,7.00,2,8,4,0.165,0,0,0,0,2,160,1,0


***Australian Dataset Parameters***

In [12]:
print("\n----- Australian Dataset Parameters -----")
print(f"Dimension Layout (Rows x Cols): {df_australian.shape}")
print(f"Distribution Vector (Col 14): {df_australian.iloc[:, -1].value_counts().to_dict()}")
print(f"Aggregated Missing Anomalies: {df_australian.isna().sum().sum()}")
display(df_australian.head(2))


----- Australian Dataset Parameters -----
Dimension Layout (Rows x Cols): (690, 15)
Distribution Vector (Col 14): {0: 383, 1: 307}
Aggregated Missing Anomalies: 0


,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14
0,1,22.08,11.46,2,4,4,1.585,0,0,0,1,2,100,1213,0
1,0,22.67,7.00,2,8,4,0.165,0,0,0,0,2,160,1,0


## ***Preprocessing Formulations & Safe-Fold Transformations***

### ***Australian Dataset Preprocessing Details***

**Categorical Column Resolution:**
The paper's Table 1 lists Australian dataset features but contains semantic inconsistencies with the official UCI repository documentation. We resolve this as follows:

***Official UCI Specification:***
- Categorical columns (indices): [0, 3, 4, 5, 8, 9, 11]
- Continuous columns (indices): [1, 2, 6, 7, 10, 12, 13]

***Implementation Decision:***
We use the ***official UCI specification*** rather than the paper's description because:
1. UCI repository is the authoritative source
2. Paper's Table 1 appears to conflate German and Australian features
3. Using official spec ensures reproducibility for future researchers

***Preprocessing Pipeline:***
1. ***Categorical features (indices 0, 3, 4, 5, 8, 9, 11):***
   - Imputation: Mode (most frequent value)
   - Encoding: OrdinalEncoder (integer mapping)
2. ***Continuous features (indices 1, 2, 6, 7, 10, 12, 13):***
   - Imputation: Mean
   - Scaling: StandardScaler (zero mean, unit variance)
3. ***Missing value handling:***
   - Strategy: Impute cleanly before scaling/encoding inside the active fold.

***Reproducibility Note:*** This discrepancy between paper description and official dataset limits is documented formally as a reproducibility challenge.


***Function To Process German Dataset***

In [13]:
def PreprocessGermanFold(x_train, x_val):
    scaler = StandardScaler()
    x_train_scaled = scaler.fit_transform(x_train)
    x_val_scaled = scaler.transform(x_val)
    return x_train_scaled, x_val_scaled

***Function To Process Australian Dataset***

In [14]:
def PreprocessAustralianFold(x_train, x_val):
    categorical_cols = [0, 3, 4, 5, 8, 9, 11]
    numerical_cols = [1, 2, 6, 7, 10, 12, 13]
    
    categorical_cols = [col for col in categorical_cols if col < x_train.shape[1]]
    numerical_cols = [col for col in numerical_cols if col < x_train.shape[1]]

    num_pipeline = Pipeline([
        ('imputer', SimpleImputer(strategy='mean')),
        ('scaler', StandardScaler())
    ])
    
    cat_pipeline = Pipeline([
        ('imputer', SimpleImputer(strategy='most_frequent')),
        ('encoder', OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)),
        ('scaler', StandardScaler()) 
    ])

    preprocessor = ColumnTransformer([
        ('num', num_pipeline, numerical_cols),
        ('cat', cat_pipeline, categorical_cols)
    ])
    
    x_train_processed = preprocessor.fit_transform(x_train)
    x_val_processed = preprocessor.transform(x_val)
    
    return x_train_processed, x_val_processed


## ***Metrics Derivations***
***Rationale:*** Standard computational bounds collapse to `0` / `1` targets tracking strict Specificity formulas.


In [15]:
def MapGermanLabels(y_vector):
    return np.where(y_vector == 1, 0, 1)

def MapAustralianLabels(y_vector):
    if isinstance(y_vector, pd.Series):
        return y_vector.values
    return y_vector

def ComputeMetrics(y_true, y_pred):
    accuracy = accuracy_score(y_true, y_pred)
    sensitivity = recall_score(y_true, y_pred, zero_division=0)
    
    conf_matrix = confusion_matrix(y_true, y_pred)
    if conf_matrix.shape == (2, 2):
        tn, fp, fn, tp = conf_matrix.ravel()
        specificity = tn / (tn + fp) if (tn + fp) > 0 else 0.0
    else:
        specificity = 0.0 if len(np.unique(y_true)) > 1 else 1.0
        
    return {
        'accuracy': accuracy, 
        'sensitivity': sensitivity, 
        'specificity': specificity
    }


## ***SMOTE-ENN Resampling Constraints***


In [16]:
def ApplySmoteEnn(x_train, y_train):
    from imblearn.over_sampling import SMOTE
    from imblearn.under_sampling import EditedNearestNeighbours
    # 'auto' safely handles near-balanced datasets like Australian without crashing
    smote = SMOTE(sampling_strategy='auto', random_state=42)
    # Soften ENN to retain hard negatives by using 'mode' instead of unanimous voting
    enn = EditedNearestNeighbours(n_neighbors=5, kind_sel='mode')
    smote_enn_engine = SMOTEENN(smote=smote, enn=enn, random_state=42)
    x_resampled, y_resampled = smote_enn_engine.fit_resample(x_train, y_train)
    return x_resampled, y_resampled


## ***Tensor Converters***


In [17]:
def BuildTensorDatasets(x_array, y_array):
    tensor_x = torch.tensor(x_array, dtype=torch.float32)
    tensor_y = torch.tensor(y_array, dtype=torch.float32).unsqueeze(1) 
    return TensorDataset(tensor_x, tensor_y)

def BuildDataLoaders(train_dataset, val_dataset, load_batch_size):
    loader_train = DataLoader(
        train_dataset, batch_size=load_batch_size, shuffle=True, 
        num_workers=num_workers, pin_memory=pin_memory
    )
    loader_val = DataLoader(
        val_dataset, batch_size=load_batch_size, shuffle=False, 
        num_workers=num_workers, pin_memory=pin_memory
    )
    return loader_train, loader_val

## ***Hyperparameter Configuration Table & Model Tracking Modules***

Our implementation strictly applies the optimal ranges designated directly by the paper's original definitions.

| Hyperparameter | MLP | LSTM | GRU | Source |
|----------------|-----|------|-----|--------|
| **Learning Rate** | 0.001 | 0.001 | 0.001 | Paper Table 3 |
| **Batch Size** | 32 | 64 | 64 | Paper Table 3 |
| **Hidden Units** | 128 | 256 | 256 | Paper Table 3 |
| **Dropout Rate** | 0.3 | 0.3 | 0.3 | Paper Table 3 |
| **Optimizer** | Adam | RMSprop | Adam | Paper Table 3 |
| **Epochs** | 100 | 100 | 100 | Paper Table 3 |
| **Weight Decay** | 0.0 | 0.0 | 0.0 | Default (not formally specified) |
| **Loss Function** | BCEWithLogitsLoss | BCEWithLogitsLoss | BCEWithLogitsLoss | Formal metric |

***Sequence Shaping Framework (LSTM/GRU):***
- Input Shape Requirement: `(batch_size, sequence_length=1, num_features)`
- Rationale: Tabular arrays cross-sections fundamentally omit embedded sequential attributes natively. A temporal vector space mapped cleanly to local depth representations represents the highest standard format.


In [18]:
class BuildMlpModel(nn.Module):
    def __init__(self, in_features_dim):
        super().__init__()
        self.dense_network = nn.Sequential(
            nn.Linear(in_features_dim, 128),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128, 128),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128, 1)
        )
    def forward(self, input_tensor):
        return self.dense_network(input_tensor)

class BuildLstmModel(nn.Module):
    def __init__(self, in_features_dim):
        super().__init__()
        self.lstm_layer = nn.LSTM(input_size=in_features_dim, hidden_size=256, batch_first=True)
        self.dropout_layer = nn.Dropout(0.3)
        self.output_dense = nn.Linear(256, 1)
    def forward(self, input_tensor):
        series_stack = input_tensor.unsqueeze(1) 
        output_states, (hidden, cell) = self.lstm_layer(series_stack)
        dropped_out = self.dropout_layer(output_states[:, -1, :]) 
        return self.output_dense(dropped_out)

class BuildGruModel(nn.Module):
    def __init__(self, in_features_dim):
        super().__init__()
        self.gru_layer = nn.GRU(input_size=in_features_dim, hidden_size=256, batch_first=True)
        self.dropout_layer = nn.Dropout(0.3)
        self.output_dense = nn.Linear(256, 1)
    def forward(self, input_tensor):
        series_stack = input_tensor.unsqueeze(1)
        output_states, hidden_state = self.gru_layer(series_stack)
        dropped_out = self.dropout_layer(output_states[:, -1, :])
        return self.output_dense(dropped_out)

def GetModelAndOptimizer(model_label, in_features_dim):
    if model_label == 'MLP':
        neural_model = BuildMlpModel(in_features_dim)
        adam_optimizer = optim.Adam(neural_model.parameters(), lr=0.001)
        batch_limit = 32
    elif model_label == 'LSTM':
        neural_model = BuildLstmModel(in_features_dim)
        adam_optimizer = optim.RMSprop(neural_model.parameters(), lr=0.001)
        batch_limit = 64
    elif model_label == 'GRU':
        neural_model = BuildGruModel(in_features_dim)
        adam_optimizer = optim.Adam(neural_model.parameters(), lr=0.001)
        batch_limit = 64
    else:
        raise ValueError("Critical: Unrecognized model variant.")
    
    if device_count > 1:
        neural_model = nn.DataParallel(neural_model)
    neural_model = neural_model.to(device)
    return neural_model, adam_optimizer, batch_limit

## ***Core Training Loop Execution Framework***


In [19]:
def TrainOneEpoch(neural_model, data_loader, loss_criterion, iteration_optimizer, gradient_scaler):
    neural_model.train()
    tracked_loss = 0.0
    for local_x, local_y in data_loader:
        local_x, local_y = local_x.to(device), local_y.to(device)
        
        iteration_optimizer.zero_grad()
        with torch.cuda.amp.autocast(enabled=device.type=='cuda'):
            raw_outputs = neural_model(local_x)
            gradient_loss = loss_criterion(raw_outputs, local_y)
            
        gradient_scaler.scale(gradient_loss).backward()
        gradient_scaler.step(iteration_optimizer)
        gradient_scaler.update()
        tracked_loss += gradient_loss.item() * local_x.size(0)
    return tracked_loss / len(data_loader.dataset)

def EvaluateOneEpoch(neural_model, data_loader, loss_criterion):
    neural_model.eval()
    tracked_loss = 0.0
    accumulated_preds = []
    accumulated_actuals = []
    with torch.no_grad():
        for local_x, local_y in data_loader:
            local_x, local_y = local_x.to(device), local_y.to(device)
            with torch.cuda.amp.autocast(enabled=device.type=='cuda'):
                raw_outputs = neural_model(local_x)
                gradient_loss = loss_criterion(raw_outputs, local_y)
            tracked_loss += gradient_loss.item() * local_x.size(0)
            
            sigmoids = torch.sigmoid(raw_outputs)
            thresholds = (sigmoids > 0.5).float()
            accumulated_preds.extend(thresholds.cpu().numpy())
            accumulated_actuals.extend(local_y.cpu().numpy())
            
    epoch_val_loss = tracked_loss / len(data_loader.dataset)
    computed_metrics = ComputeMetrics(np.array(accumulated_actuals), np.array(accumulated_preds))
    computed_metrics['loss'] = epoch_val_loss
    return computed_metrics

## ***Cross Validation Strategy, TQDM Embeddings, and Master Tracking Engine***
***Purpose***: Run dynamic constraints managing memory effectively per iteration bounds isolating memory artifacts across nodes efficiently deploying exact logging protocols universally.


In [20]:
def FitModel(neural_model, loader_train, loader_val, iteration_optimizer, fold_idx, dataset_name, model_name, logger_obj, num_epochs=100):
    loss_criterion = nn.BCEWithLogitsLoss()
    gradient_scaler = torch.cuda.amp.GradScaler(enabled=device.type=='cuda')
    
    history_tracker = {
        'train_losses': [], 'val_losses': [], 
        'val_accuracies': [], 'val_sensitivities': [], 'val_specificities': []
    }
    peak_accuracy = -1
    isolated_state_dict = None
    
    epoch_pbar = tqdm(range(num_epochs), desc=f"Fold {fold_idx+1}", leave=False)
    for iteration_step in epoch_pbar:
        loss_train = TrainOneEpoch(neural_model, loader_train, loss_criterion, iteration_optimizer, gradient_scaler)
        metrics_val = EvaluateOneEpoch(neural_model, loader_val, loss_criterion)
        
        history_tracker['train_losses'].append(loss_train)
        history_tracker['val_losses'].append(metrics_val['loss'])
        history_tracker['val_accuracies'].append(metrics_val['accuracy'])
        history_tracker['val_sensitivities'].append(metrics_val['sensitivity'])
        history_tracker['val_specificities'].append(metrics_val['specificity'])
        
        if metrics_val['accuracy'] > peak_accuracy:
            peak_accuracy = metrics_val['accuracy']
            isolated_state_dict = copy.deepcopy(neural_model.state_dict())
            
        epoch_pbar.set_postfix({'train_loss': f'{loss_train:.4f}', 'val_acc': f"{metrics_val['accuracy']:.4f}"})
        logger_obj.info(f"Epoch {iteration_step+1}/{num_epochs} - Train Loss: {loss_train:.4f} - Val Acc: {metrics_val['accuracy']:.4f}")
            
    return history_tracker, isolated_state_dict

def RunCrossValidationExperiment(dataset_name, model_name, x_data, y_data, use_smote_enn, epochs=100, n_folds=10, batch_size=32):
    condition_label = "smoteenn" if use_smote_enn else "baseline"
    
    fold_driver = StratifiedKFold(n_splits=n_folds, shuffle=True, random_state=42)
    metric_bounds = []
    historical_sequences = []
    
    global_top_accuracy = -1
    global_top_state = None
    global_top_fold = -1
    global_width = 0 
    best_x_val_stack = None
    
    main_pbar = tqdm(fold_driver.split(x_data, y_data), total=n_folds, desc=f"{dataset_name.capitalize()} | {model_name} | {condition_label.upper()}")
    
    for fold_step, (t_idx, v_idx) in enumerate(main_pbar):
        log_file = f"results/logs/{dataset_name}_{model_name}_{condition_label}_fold{fold_step}.log"
        file_handler = logging.FileHandler(log_file, mode='w')
        file_handler.setLevel(logging.INFO)
        logger.addHandler(file_handler)
        
        x_tr, x_val = x_data.iloc[t_idx].values, x_data.iloc[v_idx].values
        y_tr, y_val = y_data[t_idx], y_data[v_idx]
        
        logger.info(f"Original class distribution: {np.bincount(y_tr)}")
        logger.info(f"Training device bound securely to: {device}")
        
        if dataset_name == 'australian':
            x_tr_processed, x_val_processed = PreprocessAustralianFold(x_tr, x_val)
        else:
            x_tr_processed, x_val_processed = PreprocessGermanFold(x_tr, x_val)
            
        global_width = x_tr_processed.shape[1]
        
        if use_smote_enn:
            x_tr_processed, y_tr = ApplySmoteEnn(x_tr_processed, y_tr)
            logger.info(f"After SMOTE-ENN deployment: {np.bincount(y_tr)}")
            
        tensor_tr = BuildTensorDatasets(x_tr_processed, y_tr)
        tensor_val = BuildTensorDatasets(x_val_processed, y_val)
        
        neural_model, adam_optimizer, b_limit = GetModelAndOptimizer(model_label=model_name, in_features_dim=global_width)
        loader_train, loader_val = BuildDataLoaders(tensor_tr, tensor_val, load_batch_size=batch_size)
        
        history_tracker, local_top_state = FitModel(neural_model, loader_train, loader_val, adam_optimizer, fold_step, dataset_name, model_name, logger, num_epochs=epochs)
        
        neural_model.load_state_dict(local_top_state)
        fold_final_metrics = EvaluateOneEpoch(neural_model, loader_val, nn.BCEWithLogitsLoss())
        
        metric_bounds.append(fold_final_metrics)
        historical_sequences.append(history_tracker)
        
        if fold_final_metrics['accuracy'] > global_top_accuracy:
            global_top_accuracy = fold_final_metrics['accuracy']
            global_top_state = copy.deepcopy(local_top_state)
            global_top_fold = fold_step
            best_x_val_stack = x_val_processed
            
        # Clean nodes up per requirement
        del neural_model, adam_optimizer, loader_train, loader_val
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        gc.collect()
        logger.info(f"Fold {fold_step+1} logically complete. Tensor memory bounds cleared cleanly.")
        
        logger.removeHandler(file_handler)
        file_handler.close()

    acc_mean = np.mean([f['accuracy'] for f in metric_bounds])
    sens_mean = np.mean([f['sensitivity'] for f in metric_bounds])
    spec_mean = np.mean([f['specificity'] for f in metric_bounds])
    
    # Save explicit histories 
    df_histories = []
    for f_idx, history_dict in enumerate(historical_sequences):
        df_h = pd.DataFrame(history_dict)
        df_h['Fold'] = f_idx
        df_histories.append(df_h)
    pd.concat(df_histories).to_csv(f"results/histories/{dataset_name}_{model_name}_{condition_label}_full_history.csv", index=False)
    
    return {
        'model': model_name,
        'smoteenn': use_smote_enn,
        'mean_accuracy': acc_mean, 'std_accuracy': np.std([f['accuracy'] for f in metric_bounds]),
        'mean_sensitivity': sens_mean, 'std_sensitivity': np.std([f['sensitivity'] for f in metric_bounds]),
        'mean_specificity': spec_mean, 'std_specificity': np.std([f['specificity'] for f in metric_bounds]),
        'fold_metrics': metric_bounds,
        'fold_histories': historical_sequences,
        'best_state': global_top_state,
        'best_fold_idx': global_top_fold,
        'input_dim': global_width,
        'x_val': best_x_val_stack
    }


## ***German Dataset Experimental Execution***

***Purpose:*** Execute complete 10-fold cross-validation for all three models (MLP, LSTM, GRU) under both baseline and SMOTE-ENN conditions on the German credit dataset.

***Expected Duration:*** ~6-8 hours on Kaggle T4 x2 (can be reduced to ~2 hours with 20 epochs for testing). Currently scaled against `TESTING_MODE` limitations if defined.


In [21]:
logger.info("="*80)
logger.info("BEGINNING GERMAN DATASET EXPERIMENTAL PHASE")
logger.info("="*80)

x_german = df_german.iloc[:, :-1]
y_german = MapGermanLabels(df_german.iloc[:, -1].values)

DATASET_NAME = 'german'
MODELS_TO_TEST = ['MLP', 'LSTM', 'GRU']
RESAMPLING_CONDITIONS = [False, True] 

german_results = {}
best_german_shap_pkg = {'acc': -1}

for model_name in MODELS_TO_TEST:
    for use_smote in RESAMPLING_CONDITIONS:
        condition_name = 'smoteenn' if use_smote else 'baseline'
        experiment_key = f"{model_name}_{condition_name}"
        
        logger.info(f"\nEXPERIMENT: {experiment_key.upper()} | {DATASET_NAME} | SMOTE-ENN: {use_smote}")
        
        results_dict = RunCrossValidationExperiment(
            dataset_name=DATASET_NAME,
            model_name=model_name,
            x_data=x_german,
            y_data=y_german,
            use_smote_enn=use_smote,
            n_folds=10,
            epochs=NUM_EPOCHS,
            batch_size=32 if model_name == 'MLP' else 64
        )
        
        german_results[experiment_key] = results_dict
        
        logger.info(f"✓ Completed {experiment_key}")
        logger.info(f"  Mean Accuracy: {results_dict['mean_accuracy']:.4f} ± {results_dict['std_accuracy']:.4f}")
        
        if results_dict['mean_accuracy'] > best_german_shap_pkg['acc']:
            best_german_shap_pkg = {
                'acc': results_dict['mean_accuracy'],
                'model_name': model_name,
                'state': results_dict['best_state'],
                'input_dim': results_dict['input_dim'],
                'x_val': results_dict['x_val']
            }

german_summary_df = pd.DataFrame({
    'Experiment': list(german_results.keys()),
    'Mean_Accuracy': [german_results[k]['mean_accuracy'] for k in german_results.keys()],
    'Std_Accuracy': [german_results[k]['std_accuracy'] for k in german_results.keys()],
    'Mean_Sensitivity': [german_results[k]['mean_sensitivity'] for k in german_results.keys()],
    'Std_Sensitivity': [german_results[k]['std_sensitivity'] for k in german_results.keys()],
    'Mean_Specificity': [german_results[k]['mean_specificity'] for k in german_results.keys()],
    'Std_Specificity': [german_results[k]['std_specificity'] for k in german_results.keys()]
})

german_summary_df.to_csv('results/metrics/german_summary_results.csv', index=False)
print("\nGERMAN DATASET - FINAL RESULTS SUMMARY")
display(german_summary_df)


2026-04-05 05:15:20,260 - INFO - ================================================================================
2026-04-05 05:15:20,261 - INFO - BEGINNING GERMAN DATASET EXPERIMENTAL PHASE
2026-04-05 05:15:20,262 - INFO - ================================================================================
2026-04-05 05:15:20,266 - INFO - 
EXPERIMENT: MLP_BASELINE | german | SMOTE-ENN: False


German | MLP | BASELINE:   0%|          | 0/10 [00:00<?, ?it/s]

2026-04-05 05:15:20,285 - INFO - Original class distribution: [277 344]
2026-04-05 05:15:20,285 - INFO - Training device bound securely to: cuda


Fold 1:   0%|          | 0/100 [00:00<?, ?it/s]

2026-04-05 05:15:27,160 - INFO - Epoch 1/100 - Train Loss: 0.6252 - Val Acc: 0.8841
2026-04-05 05:15:27,461 - INFO - Epoch 2/100 - Train Loss: 0.4513 - Val Acc: 0.8841
2026-04-05 05:15:27,776 - INFO - Epoch 3/100 - Train Loss: 0.3711 - Val Acc: 0.8841
2026-04-05 05:15:28,090 - INFO - Epoch 4/100 - Train Loss: 0.3447 - Val Acc: 0.8986
2026-04-05 05:15:28,400 - INFO - Epoch 5/100 - Train Loss: 0.3315 - Val Acc: 0.8986
2026-04-05 05:15:28,713 - INFO - Epoch 6/100 - Train Loss: 0.3145 - Val Acc: 0.8986
2026-04-05 05:15:29,017 - INFO - Epoch 7/100 - Train Loss: 0.3062 - Val Acc: 0.8986
2026-04-05 05:15:29,327 - INFO - Epoch 8/100 - Train Loss: 0.3078 - Val Acc: 0.8841
2026-04-05 05:15:29,637 - INFO - Epoch 9/100 - Train Loss: 0.3106 - Val Acc: 0.8841
2026-04-05 05:15:29,945 - INFO - Epoch 10/100 - Train Loss: 0.3025 - Val Acc: 0.8841
2026-04-05 05:15:30,265 - INFO - Epoch 11/100 - Train Loss: 0.2862 - Val Acc: 0.8841
2026-04-05 05:15:30,578 - INFO - Epoch 12/100 - Train Loss: 0.2902 - Val A

...

2026-04-05 05:43:50,369 - INFO - Epoch 1/100 - Train Loss: 0.6520 - Val Acc: 0.7681
2026-04-05 05:43:50,648 - INFO - Epoch 2/100 - Train Loss: 0.5660 - Val Acc: 0.7536
2026-04-05 05:43:50,919 - INFO - Epoch 3/100 - Train Loss: 0.4836 - Val Acc: 0.7536
2026-04-05 05:43:51,186 - INFO - Epoch 4/100 - Train Loss: 0.4113 - Val Acc: 0.7681
2026-04-05 05:43:51,447 - INFO - Epoch 5/100 - Train Loss: 0.3506 - Val Acc: 0.7826
2026-04-05 05:43:51,722 - INFO - Epoch 6/100 - Train Loss: 0.3076 - Val Acc: 0.7971
2026-04-05 05:43:51,996 - INFO - Epoch 7/100 - Train Loss: 0.2799 - Val Acc: 0.8116
2026-04-05 05:43:52,261 - INFO - Epoch 8/100 - Train Loss: 0.2577 - Val Acc: 0.7971
2026-04-05 05:43:52,545 - INFO - Epoch 9/100 - Train Loss: 0.2392 - Val Acc: 0.7971
2026-04-05 05:43:52,818 - INFO - Epoch 10/100 - Train Loss: 0.2237 - Val Acc: 0.8116
2026-04-05 05:43:53,084 - INFO - Epoch 11/100 - Train Loss: 0.2141 - Val Acc: 0.8116
2026-04-05 05:43:53,360 - INFO - Epoch 12/100 - Train Loss: 0.2033 - Val A

Fold 10:   0%|          | 0/100 [00:00<?, ?it/s]

2026-04-05 05:44:18,031 - INFO - Epoch 1/100 - Train Loss: 0.6522 - Val Acc: 0.8116
2026-04-05 05:44:18,296 - INFO - Epoch 2/100 - Train Loss: 0.5709 - Val Acc: 0.8261
2026-04-05 05:44:18,564 - INFO - Epoch 3/100 - Train Loss: 0.4877 - Val Acc: 0.8261
2026-04-05 05:44:18,841 - INFO - Epoch 4/100 - Train Loss: 0.4120 - Val Acc: 0.8406
2026-04-05 05:44:19,099 - INFO - Epoch 5/100 - Train Loss: 0.3558 - Val Acc: 0.8406
2026-04-05 05:44:19,360 - INFO - Epoch 6/100 - Train Loss: 0.3133 - Val Acc: 0.8841
2026-04-05 05:44:19,619 - INFO - Epoch 7/100 - Train Loss: 0.2870 - Val Acc: 0.8841
2026-04-05 05:44:19,879 - INFO - Epoch 8/100 - Train Loss: 0.2660 - Val Acc: 0.8986
2026-04-05 05:44:20,142 - INFO - Epoch 9/100 - Train Loss: 0.2507 - Val Acc: 0.9275
2026-04-05 05:44:20,413 - INFO - Epoch 10/100 - Train Loss: 0.2386 - Val Acc: 0.9275
2026-04-05 05:44:20,693 - INFO - Epoch 11/100 - Train Loss: 0.2310 - Val Acc: 0.9420
2026-04-05 05:44:20,957 - INFO - Epoch 12/100 - Train Loss: 0.2206 - Val A


GERMAN DATASET - FINAL RESULTS SUMMARY


,Experiment,Mean_Accuracy,Std_Accuracy,Mean_Sensitivity,Std_Sensitivity,Mean_Specificity,Std_Specificity
0,MLP_baseline,0.884058,0.048933,0.903576,0.046483,0.861075,0.108754
1,MLP_smoteenn,0.875362,0.047716,0.840823,0.057802,0.919247,0.080581
2,LSTM_baseline,0.891304,0.041120,0.919366,0.034981,0.857849,0.105066
3,LSTM_smoteenn,0.879710,0.047207,0.845951,0.058107,0.922473,0.070900
4,GRU_baseline,0.894203,0.040502,0.919366,0.034981,0.864301,0.103684
5,GRU_smoteenn,0.876812,0.047296,0.845951,0.054415,0.916022,0.080512


## ***Australian Dataset Experimental Execution***

***Purpose:*** Execute complete 10-fold cross-validation natively deploying dataset-specific preprocessing mechanisms (mode/mean + numerical boundaries strictly resolved) as implemented previously inside Section 6.


In [22]:
logger.info("="*80)
logger.info("BEGINNING AUSTRALIAN DATASET EXPERIMENTAL PHASE")
logger.info("="*80)

x_australian = df_australian.iloc[:, :-1]
y_australian = MapAustralianLabels(df_australian.iloc[:, -1].values)

DATASET_NAME = 'australian'

australian_results = {}
best_australian_shap_pkg = {'acc': -1}

for model_name in MODELS_TO_TEST:
    for use_smote in RESAMPLING_CONDITIONS:
        condition_name = 'smoteenn' if use_smote else 'baseline'
        experiment_key = f"{model_name}_{condition_name}"
        
        logger.info(f"\nEXPERIMENT: {experiment_key.upper()} | {DATASET_NAME} | SMOTE-ENN: {use_smote}")
        
        results_dict = RunCrossValidationExperiment(
            dataset_name=DATASET_NAME,
            model_name=model_name,
            x_data=x_australian,
            y_data=y_australian,
            use_smote_enn=use_smote,
            n_folds=10,
            epochs=NUM_EPOCHS,
            batch_size=32 if model_name == 'MLP' else 64
        )
        
        australian_results[experiment_key] = results_dict
        logger.info(f"✓ Completed {experiment_key} -> Acc: {results_dict['mean_accuracy']:.4f}")
        
        if results_dict['mean_accuracy'] > best_australian_shap_pkg['acc']:
            best_australian_shap_pkg = {
                'acc': results_dict['mean_accuracy'],
                'model_name': model_name,
                'state': results_dict['best_state'],
                'input_dim': results_dict['input_dim'],
                'x_val': results_dict['x_val']
            }

australian_summary_df = pd.DataFrame({
    'Experiment': list(australian_results.keys()),
    'Mean_Accuracy': [australian_results[k]['mean_accuracy'] for k in australian_results.keys()],
    'Std_Accuracy': [australian_results[k]['std_accuracy'] for k in australian_results.keys()],
    'Mean_Sensitivity': [australian_results[k]['mean_sensitivity'] for k in australian_results.keys()],
    'Std_Sensitivity': [australian_results[k]['std_sensitivity'] for k in australian_results.keys()],
    'Mean_Specificity': [australian_results[k]['mean_specificity'] for k in australian_results.keys()],
    'Std_Specificity': [australian_results[k]['std_specificity'] for k in australian_results.keys()]
})

australian_summary_df.to_csv('results/metrics/australian_summary_results.csv', index=False)
print("\nAUSTRALIAN DATASET - FINAL RESULTS SUMMARY")
display(australian_summary_df)


2026-04-05 05:44:45,205 - INFO - ================================================================================
2026-04-05 05:44:45,206 - INFO - BEGINNING AUSTRALIAN DATASET EXPERIMENTAL PHASE
2026-04-05 05:44:45,207 - INFO - ================================================================================
2026-04-05 05:44:45,209 - INFO - 
EXPERIMENT: MLP_BASELINE | australian | SMOTE-ENN: False


Australian | MLP | BASELINE:   0%|          | 0/10 [00:00<?, ?it/s]

2026-04-05 05:44:45,224 - INFO - Original class distribution: [344 277]
2026-04-05 05:44:45,224 - INFO - Training device bound securely to: cuda


Fold 1:   0%|          | 0/100 [00:00<?, ?it/s]

2026-04-05 05:44:45,579 - INFO - Epoch 1/100 - Train Loss: 0.6119 - Val Acc: 0.8696
2026-04-05 05:44:45,897 - INFO - Epoch 2/100 - Train Loss: 0.4408 - Val Acc: 0.8986
2026-04-05 05:44:46,216 - INFO - Epoch 3/100 - Train Loss: 0.3683 - Val Acc: 0.8986
2026-04-05 05:44:46,538 - INFO - Epoch 4/100 - Train Loss: 0.3184 - Val Acc: 0.9130
2026-04-05 05:44:46,852 - INFO - Epoch 5/100 - Train Loss: 0.3213 - Val Acc: 0.9130
2026-04-05 05:44:47,170 - INFO - Epoch 6/100 - Train Loss: 0.3073 - Val Acc: 0.8986
2026-04-05 05:44:47,482 - INFO - Epoch 7/100 - Train Loss: 0.3172 - Val Acc: 0.8986
2026-04-05 05:44:47,810 - INFO - Epoch 8/100 - Train Loss: 0.3078 - Val Acc: 0.8841
2026-04-05 05:44:48,138 - INFO - Epoch 9/100 - Train Loss: 0.3053 - Val Acc: 0.8841
2026-04-05 05:44:48,456 - INFO - Epoch 10/100 - Train Loss: 0.2910 - Val Acc: 0.8986
2026-04-05 05:44:48,775 - INFO - Epoch 11/100 - Train Loss: 0.2916 - Val Acc: 0.8986
2026-04-05 05:44:49,084 - INFO - Epoch 12/100 - Train Loss: 0.2769 - Val A

...

2026-04-05 06:13:29,190 - INFO - Epoch 1/100 - Train Loss: 0.6467 - Val Acc: 0.7826
2026-04-05 06:13:29,473 - INFO - Epoch 2/100 - Train Loss: 0.5482 - Val Acc: 0.8116
2026-04-05 06:13:29,757 - INFO - Epoch 3/100 - Train Loss: 0.4563 - Val Acc: 0.7971
2026-04-05 06:13:30,039 - INFO - Epoch 4/100 - Train Loss: 0.3821 - Val Acc: 0.8116
2026-04-05 06:13:30,312 - INFO - Epoch 5/100 - Train Loss: 0.3274 - Val Acc: 0.8116
2026-04-05 06:13:30,591 - INFO - Epoch 6/100 - Train Loss: 0.2918 - Val Acc: 0.8261
2026-04-05 06:13:30,878 - INFO - Epoch 7/100 - Train Loss: 0.2687 - Val Acc: 0.8406
2026-04-05 06:13:31,185 - INFO - Epoch 8/100 - Train Loss: 0.2524 - Val Acc: 0.8406
2026-04-05 06:13:31,464 - INFO - Epoch 9/100 - Train Loss: 0.2401 - Val Acc: 0.8406
2026-04-05 06:13:31,745 - INFO - Epoch 10/100 - Train Loss: 0.2289 - Val Acc: 0.8406
2026-04-05 06:13:32,033 - INFO - Epoch 11/100 - Train Loss: 0.2180 - Val Acc: 0.8261
2026-04-05 06:13:32,309 - INFO - Epoch 12/100 - Train Loss: 0.2125 - Val A

Fold 10:   0%|          | 0/100 [00:00<?, ?it/s]

2026-04-05 06:13:58,005 - INFO - Epoch 1/100 - Train Loss: 0.6497 - Val Acc: 0.8986
2026-04-05 06:13:58,279 - INFO - Epoch 2/100 - Train Loss: 0.5572 - Val Acc: 0.8986
2026-04-05 06:13:58,555 - INFO - Epoch 3/100 - Train Loss: 0.4732 - Val Acc: 0.8986
2026-04-05 06:13:58,840 - INFO - Epoch 4/100 - Train Loss: 0.4041 - Val Acc: 0.8986
2026-04-05 06:13:59,116 - INFO - Epoch 5/100 - Train Loss: 0.3480 - Val Acc: 0.8986
2026-04-05 06:13:59,398 - INFO - Epoch 6/100 - Train Loss: 0.3088 - Val Acc: 0.8986
2026-04-05 06:13:59,668 - INFO - Epoch 7/100 - Train Loss: 0.2781 - Val Acc: 0.8986
2026-04-05 06:13:59,940 - INFO - Epoch 8/100 - Train Loss: 0.2598 - Val Acc: 0.9130
2026-04-05 06:14:00,216 - INFO - Epoch 9/100 - Train Loss: 0.2468 - Val Acc: 0.9130
2026-04-05 06:14:00,494 - INFO - Epoch 10/100 - Train Loss: 0.2406 - Val Acc: 0.9130
2026-04-05 06:14:00,778 - INFO - Epoch 11/100 - Train Loss: 0.2378 - Val Acc: 0.9130
2026-04-05 06:14:01,067 - INFO - Epoch 12/100 - Train Loss: 0.2313 - Val A


AUSTRALIAN DATASET - FINAL RESULTS SUMMARY


,Experiment,Mean_Accuracy,Std_Accuracy,Mean_Sensitivity,Std_Sensitivity,Mean_Specificity,Std_Specificity
0,MLP_baseline,0.892754,0.041602,0.873871,0.093747,0.908839,0.057389
1,MLP_smoteenn,0.881159,0.050952,0.818817,0.113323,0.932186,0.028979
2,LSTM_baseline,0.894203,0.038372,0.867419,0.093875,0.916734,0.035737
3,LSTM_smoteenn,0.881159,0.052174,0.818925,0.122233,0.932389,0.032720
4,GRU_baseline,0.891304,0.038481,0.854624,0.102096,0.921997,0.034006
5,GRU_smoteenn,0.881159,0.057899,0.815591,0.121961,0.934885,0.030972


## ***Reproduced Results vs. Paper Comparison***
***Purpose:*** Generate comprehensive comparison tables structurally showing our reproduced explicit metric sets mapped natively identical against the original paper's definitively reported data tables.

*Paper Citation: Idowu Aruleba and Yanxia Sun, "Enhanced credit risk prediction using deep learning and SMOTE-ENN resampling", Machine Learning with Applications, vol. 21, 2025.*


In [23]:
logger.info("="*80)
logger.info("GENERATING PAPER VS REPRODUCED COMPARISON TABLES")
logger.info("="*80)

german_paper_results = {
    'MLP_before': {'accuracy': 0.704, 'sensitivity': 0.773, 'specificity': 0.310},
    'MLP_after': {'accuracy': 0.824, 'sensitivity': 0.830, 'specificity': 0.810},
    'LSTM_before': {'accuracy': 0.730, 'sensitivity': 0.770, 'specificity': 0.394},
    'LSTM_after': {'accuracy': 0.819, 'sensitivity': 0.859, 'specificity': 0.729},
    'GRU_before': {'accuracy': 0.749, 'sensitivity': 0.895, 'specificity': 0.431},
    'GRU_after': {'accuracy': 0.899, 'sensitivity': 0.931, 'specificity': 0.852}
}

german_comparison_rows = []
for model in ['MLP', 'LSTM', 'GRU']:
    for condition in ['baseline', 'smoteenn']:
        key = f"{model}_{condition}"
        paper_key = f"{model}_{'before' if condition == 'baseline' else 'after'}"
        
        our_acc, our_sens, our_spec = german_results[key]['mean_accuracy'], german_results[key]['mean_sensitivity'], german_results[key]['mean_specificity']
        paper_acc, paper_sens, paper_spec = german_paper_results[paper_key]['accuracy'], german_paper_results[paper_key]['sensitivity'], german_paper_results[paper_key]['specificity']
        
        for met_name, our_val, p_val in zip(['Accuracy', 'Sensitivity', 'Specificity'], [our_acc, our_sens, our_spec], [paper_acc, paper_sens, paper_spec]):
            diff = our_val - p_val
            german_comparison_rows.append({
                'Model': model, 'Condition': condition.upper(), 'Metric': met_name,
                'Paper': f"{p_val:.3f}", 'Reproduced': f"{our_val:.3f}",
                'Difference': f"{diff:+.3f}", 'Pct_Diff': f"{(diff/p_val*100):+.1f}%"
            })

german_comparison_df = pd.DataFrame(german_comparison_rows)
german_comparison_df.to_csv('results/tables/german_paper_vs_reproduced.csv', index=False)
print("\nGERMAN DATASET: PAPER VS REPRODUCED RESULTS")
display(german_comparison_df)

australian_paper_results = {
    'MLP_before': {'accuracy': 0.810, 'sensitivity': 0.850, 'specificity': 0.781},
    'MLP_after': {'accuracy': 0.850, 'sensitivity': 0.879, 'specificity': 0.830},
    'LSTM_before': {'accuracy': 0.861, 'sensitivity': 0.848, 'specificity': 0.885},
    'LSTM_after': {'accuracy': 0.917, 'sensitivity': 0.897, 'specificity': 0.913},
    'GRU_before': {'accuracy': 0.837, 'sensitivity': 0.829, 'specificity': 0.849},
    'GRU_after': {'accuracy': 0.926, 'sensitivity': 0.911, 'specificity': 0.930}
}

australian_comparison_rows = []
for model in ['MLP', 'LSTM', 'GRU']:
    for condition in ['baseline', 'smoteenn']:
        key = f"{model}_{condition}"
        paper_key = f"{model}_{'before' if condition == 'baseline' else 'after'}"
        
        our_acc, our_sens, our_spec = australian_results[key]['mean_accuracy'], australian_results[key]['mean_sensitivity'], australian_results[key]['mean_specificity']
        paper_acc, paper_sens, paper_spec = australian_paper_results[paper_key]['accuracy'], australian_paper_results[paper_key]['sensitivity'], australian_paper_results[paper_key]['specificity']
        
        for met_name, our_val, p_val in zip(['Accuracy', 'Sensitivity', 'Specificity'], [our_acc, our_sens, our_spec], [paper_acc, paper_sens, paper_spec]):
            diff = our_val - p_val
            australian_comparison_rows.append({
                'Model': model, 'Condition': condition.upper(), 'Metric': met_name,
                'Paper': f"{p_val:.3f}", 'Reproduced': f"{our_val:.3f}",
                'Difference': f"{diff:+.3f}", 'Pct_Diff': f"{(diff/p_val*100):+.1f}%"
            })

australian_comparison_df = pd.DataFrame(australian_comparison_rows)
australian_comparison_df.to_csv('results/tables/australian_paper_vs_reproduced.csv', index=False)
print("\nAUSTRALIAN DATASET: PAPER VS REPRODUCED RESULTS")
display(australian_comparison_df)

print("\nREPRODUCTION FIDELITY ANALYSIS")
ger_diffs = german_comparison_df['Difference'].apply(lambda x: abs(float(x)))
aus_diffs = australian_comparison_df['Difference'].apply(lambda x: abs(float(x)))
print(f"German Abs Mean Diff: {ger_diffs.mean():.4f} | Max Diff: {ger_diffs.max():.4f}")
print(f"Australian Abs Mean Diff: {aus_diffs.mean():.4f} | Max Diff: {aus_diffs.max():.4f}")

2026-04-05 06:14:26,261 - INFO - ================================================================================
2026-04-05 06:14:26,262 - INFO - GENERATING PAPER VS REPRODUCED COMPARISON TABLES
2026-04-05 06:14:26,263 - INFO - ================================================================================



GERMAN DATASET: PAPER VS REPRODUCED RESULTS


,Model,Condition,Metric,Paper,Reproduced,Difference,Pct_Diff
0,MLP,BASELINE,Accuracy,0.704,0.884,+0.180,+25.6%
1,MLP,BASELINE,Sensitivity,0.773,0.904,+0.131,+16.9%
2,MLP,BASELINE,Specificity,0.310,0.861,+0.551,+177.8%
3,MLP,SMOTEENN,Accuracy,0.824,0.875,+0.051,+6.2%
4,MLP,SMOTEENN,Sensitivity,0.830,0.841,+0.011,+1.3%
5,MLP,SMOTEENN,Specificity,0.810,0.919,+0.109,+13.5%
6,LSTM,BASELINE,Accuracy,0.730,0.891,+0.161,+22.1%
7,LSTM,BASELINE,Sensitivity,0.770,0.919,+0.149,+19.4%
8,LSTM,BASELINE,Specificity,0.394,0.858,+0.464,+117.7%
9,LSTM,SMOTEENN,Accuracy,0.819,0.880,+0.061,+7.4%



AUSTRALIAN DATASET: PAPER VS REPRODUCED RESULTS


,Model,Condition,Metric,Paper,Reproduced,Difference,Pct_Diff
0,MLP,BASELINE,Accuracy,0.810,0.893,+0.083,+10.2%
1,MLP,BASELINE,Sensitivity,0.850,0.874,+0.024,+2.8%
2,MLP,BASELINE,Specificity,0.781,0.909,+0.128,+16.4%
3,MLP,SMOTEENN,Accuracy,0.850,0.881,+0.031,+3.7%
4,MLP,SMOTEENN,Sensitivity,0.879,0.819,-0.060,-6.8%
5,MLP,SMOTEENN,Specificity,0.830,0.932,+0.102,+12.3%
6,LSTM,BASELINE,Accuracy,0.861,0.894,+0.033,+3.9%
7,LSTM,BASELINE,Sensitivity,0.848,0.867,+0.019,+2.3%
8,LSTM,BASELINE,Specificity,0.885,0.917,+0.032,+3.6%
9,LSTM,SMOTEENN,Accuracy,0.917,0.881,-0.036,-3.9%



REPRODUCTION FIDELITY ANALYSIS
German Abs Mean Diff: 0.1582 | Max Diff: 0.5510
Australian Abs Mean Diff: 0.0524 | Max Diff: 0.1280


## ***Training Dynamics Visualization***
***Purpose:*** Generate comprehensive training and validation curves physically modeling model convergence boundaries iteratively isolating absolute metrics vectors natively across individual states mapping 12 configurations.


In [24]:
def PlotTrainingCurves(results_dict, dataset_name, model_name, condition):
    fig, axes = plt.subplots(2, 2, figsize=(15, 10))
    fig.suptitle(f'{dataset_name.capitalize()} Dataset - {model_name} - {condition.upper()}\nTraining Dynamics (10-Fold CV)', fontsize=16, fontweight='bold')
    
    all_train_losses = [h['train_losses'] for h in results_dict['fold_histories']]
    all_val_losses = [h['val_losses'] for h in results_dict['fold_histories']]
    all_val_accuracies = [h['val_accuracies'] for h in results_dict['fold_histories']]
    all_val_sensitivities = [h['val_sensitivities'] for h in results_dict['fold_histories']]
    all_val_specificities = [h['val_specificities'] for h in results_dict['fold_histories']]
    
    epochs = range(1, len(all_train_losses[0]) + 1)
    
    ax = axes[0, 0]
    for losses in all_train_losses: ax.plot(epochs, losses, alpha=0.3, color='blue', linewidth=0.8)
    ax.plot(epochs, np.mean(all_train_losses, axis=0), color='darkblue', linewidth=2, label='Mean Train Loss')
    ax.set_title('Training Loss')
    ax.legend()
    
    ax = axes[0, 1]
    for losses in all_val_losses: ax.plot(epochs, losses, alpha=0.3, color='red', linewidth=0.8)
    ax.plot(epochs, np.mean(all_val_losses, axis=0), color='darkred', linewidth=2, label='Mean Val Loss')
    ax.set_title('Validation Loss')
    ax.legend()
    
    ax = axes[1, 0]
    for accs in all_val_accuracies: ax.plot(epochs, accs, alpha=0.3, color='green', linewidth=0.8)
    ax.plot(epochs, np.mean(all_val_accuracies, axis=0), color='darkgreen', linewidth=2, label='Mean Val Accuracy')
    ax.set_title('Validation Accuracy')
    ax.set_ylim([0, 1])
    ax.legend()
    
    ax = axes[1, 1]
    ax.plot(epochs, np.mean(all_val_sensitivities, axis=0), color='purple', linewidth=2, label='Mean Sensitivity')
    ax.plot(epochs, np.mean(all_val_specificities, axis=0), color='orange', linewidth=2, label='Mean Specificity')
    ax.set_title('Validation Sensitivity & Specificity')
    ax.set_ylim([0, 1])
    ax.legend()
    
    plt.tight_layout()
    base_save = f'results/plots/{dataset_name}_{model_name}_{condition}_training_curves'
    plt.savefig(f'{base_save}.png', dpi=300, bbox_inches='tight')
    plt.close()

logger.info("Generating plot clusters physically rendering limits across parameters...")
for m_name in ['MLP', 'LSTM', 'GRU']:
    for c_flag in ['baseline', 'smoteenn']:
        PlotTrainingCurves(german_results[f"{m_name}_{c_flag}"], 'german', m_name, c_flag)
        PlotTrainingCurves(australian_results[f"{m_name}_{c_flag}"], 'australian', m_name, c_flag)
logger.info("Plot artifacts rendered explicitly down to vector outputs successfully.")


2026-04-05 06:14:26,297 - INFO - Generating plot clusters physically rendering limits across parameters...
2026-04-05 06:14:43,959 - INFO - ✓ Plot artifacts rendered explicitly down to vector outputs successfully.


## ***Robust SHAP Global Implementations***
***Robust Configuration Protocols:*** Restricting the overall unrolled dimensions securely across mathematically stratified samples natively enforcing bounds guaranteeing $N \ge 20$ elements prior to dispatch.


In [25]:
def GenerateShapPlots(domain_target, best_model_pkg):
    logger.info(f"Initializing SHAP analysis for {domain_target} dataset...")
    neural_inst, _, _ = GetModelAndOptimizer(best_model_pkg['model_name'], best_model_pkg['input_dim'])
    
    state_mapping = best_model_pkg['state']
    unrolled_states = {}
    for local_k, local_v in state_mapping.items():
        stripped_key = local_k[7:] if local_k.startswith('module.') else local_k
        unrolled_states[stripped_key] = local_v
    
    if isinstance(neural_inst, nn.DataParallel):
        neural_inst.module.load_state_dict(unrolled_states)
    else:
        neural_inst.load_state_dict(unrolled_states)
    
    neural_inst.eval()
    neural_inst.to(device)
    
    x_test_subset = best_model_pkg['x_val']
    total_samples = x_test_subset.shape[0]
    
    if total_samples < 20:
        logger.warning(f"Insufficient samples ({total_samples}) for SHAP. Skipping.")
        return
        
    bg_size = min(100, total_samples // 2)
    test_size = min(50, total_samples - bg_size)
    
    np.random.seed(42)
    all_indices = np.arange(total_samples)
    np.random.shuffle(all_indices)
    
    bg_tensor = torch.tensor(x_test_subset[all_indices[:bg_size]], dtype=torch.float32).to(device)
    test_tensor = torch.tensor(x_test_subset[all_indices[bg_size:bg_size + test_size]], dtype=torch.float32).to(device)
    
    assert bg_tensor.numel() > 0, f"Background tensor is empty! Shape: {bg_tensor.shape}"
    assert test_tensor.numel() > 0, f"Test tensor is empty! Shape: {test_tensor.shape}"
    
    try:
        def model_predict(x):
            with torch.no_grad():
                if best_model_pkg['model_name'] in ['LSTM', 'GRU']:
                    output = neural_inst(x.unsqueeze(1))
                else:
                    output = neural_inst(x)
                return torch.sigmoid(output).cpu().numpy()
                
        explainer = shap.DeepExplainer(neural_inst, bg_tensor)
        shap_values = explainer.shap_values(test_tensor)
        shap_values_to_plot = shap_values[0] if isinstance(shap_values, list) else shap_values
        
        plt.figure(figsize=(12, 8))
        shap.summary_plot(shap_values_to_plot, test_tensor.cpu().numpy(), show=False, max_display=15)
        plt.title(f"SHAP Feature Importance - {domain_target.capitalize()} Dataset\nModel: {best_model_pkg['model_name']}-SMOTE", fontsize=14, fontweight='bold')
        plt.tight_layout()
        
        save_path = f'results/shap/{domain_target}_{best_model_pkg["model_name"]}_shap_summary'
        plt.savefig(f'{save_path}.png', dpi=300, bbox_inches='tight')
        plt.close()
        
        feature_names = [f"Feature_{i}" for i in range(shap_values_to_plot.shape[1])]
        pd.DataFrame(shap_values_to_plot, columns=feature_names).to_csv(f'{save_path}_values.csv', index=False)
        logger.info(f"✓ SHAP matrices cleanly exported successfully out over `{save_path}.png`")
        
    except Exception as e:
        logger.error(f"SHAP calculation mechanically blocked: {str(e)}. Bypassing visual arrays.")

GenerateShapPlots('german', best_german_shap_pkg)
GenerateShapPlots('australian', best_australian_shap_pkg)


2026-04-05 06:14:43,973 - INFO - Initializing SHAP analysis for german dataset...
2026-04-05 06:14:44,307 - ERROR - SHAP calculation mechanically blocked: cudnn RNN backward can only be called in training mode. Bypassing visual arrays.
2026-04-05 06:14:44,309 - INFO - Initializing SHAP analysis for australian dataset...
2026-04-05 06:14:44,325 - ERROR - SHAP calculation mechanically blocked: cudnn RNN backward can only be called in training mode. Bypassing visual arrays.


## ***Reproducibility Disconnection & Experimental Findings Audit***

***Quantified Discrepancy Analysis***
| Metric | Dataset | Model | Possible Causes of Deviance Arrays |
|--------|---------|-------|-----------------|
| Accuracy | German | GRU-SE | Random seeds severely effect boundaries across $N=1000$ matrices natively. |
| Sensitivity | Australian | LSTM | Categorical mappings resolving pure modes differ from author configurations strictly. |

1. ***Australian Dataset Mismatch:*** The core paper contradicts official documentation on categorical features natively mapping indices [0, 3, 4, 5, 8, 9, 11] correctly.
2. ***Tabular vs Temporal Sequence Bounds:*** GRU parameters explicitly wrap dimension vectors directly targeting $t=1$.
3. ***CNN/GNN Framework Assumptions Removed***: CNNs construct strictly over spatial bounds mathematically impossible across non-mapped flat lines. GNN variants execute against unknown defined node neighbors absent directly within standard methodology deployments. Attempting structural generations forces arbitrary unscientific parameters.


## ***Report Generation: Data Summary***
***Purpose:*** Output summary JSON dictionaries explicitly configuring parameters formally required heavily for your subsequent 10 Page deployment limits!


In [26]:
ger_b = [german_results[f'{m}_baseline']['mean_accuracy'] for m in MODELS_TO_TEST]
ger_s = [german_results[f'{m}_smoteenn']['mean_accuracy'] for m in MODELS_TO_TEST]
aus_b = [australian_results[f'{m}_baseline']['mean_accuracy'] for m in MODELS_TO_TEST]
aus_s = [australian_results[f'{m}_smoteenn']['mean_accuracy'] for m in MODELS_TO_TEST]

report_summary = {
    'german_baseline_mean': np.mean(ger_b),
    'german_smote_mean': np.mean(ger_s),
    'german_improvement': np.mean(ger_s) - np.mean(ger_b),
    'australian_baseline_mean': np.mean(aus_b),
    'australian_smote_mean': np.mean(aus_s),
    'australian_improvement': np.mean(aus_s) - np.mean(aus_b)
}

with open('results/report_summary.json', 'w') as f:
    json.dump(report_summary, f, indent=4)
logger.info("✓ Report boundaries natively dumped into root /results.")


2026-04-05 06:14:44,334 - INFO - ✓ Report boundaries natively dumped into root /results.


## ***Pre-Submission Verification Routine Block***
***Verifies physical files accurately deposited correctly over entire target sequences mapping checks explicitly ensuring complete deliverables execution.***


In [27]:
print("="*80)
print("PRE-SUBMISSION VERIFICATION CHECKLIST")
print("="*80)

checks = {'experiments_complete': True, 'comparison_tables': False, 'training_curves': False, 'shap_plots': False, 'logs_generated': False, 'csvs_exported': False}

checks['comparison_tables'] = all(os.path.exists(p) for p in ['results/tables/german_paper_vs_reproduced.csv', 'results/tables/australian_paper_vs_reproduced.csv'])

plot_dir = Path('results/plots')
if plot_dir.exists():
    checks['training_curves'] = len(list(plot_dir.glob('*training_curves.png'))) >= 12

shap_dir = Path('results/shap')
if shap_dir.exists():
    checks['shap_plots'] = len(list(shap_dir.glob('*shap_summary.png'))) >= 2

log_dir = Path('results/logs')
if log_dir.exists():
    checks['logs_generated'] = len(list(log_dir.glob('*.log'))) > 0

metrics_dir = Path('results/metrics')
if metrics_dir.exists():
    checks['csvs_exported'] = len(list(metrics_dir.glob('*.csv'))) >= 2

checks['experiments_complete'] = (german_results is not None and australian_results is not None and len(german_results) == 6 and len(australian_results) == 6)

for check_name, status in checks.items():
    print(f"{'✓' if status else '❌'} {check_name.replace('_', ' ').title()}: {'PASS' if status else 'FAIL'}")

if all(checks.values()): print("\n✓✓✓ ALL CHECKS PASSED - READY FOR KAGGLE/SUBMISSION ✓✓✓")
else: print("\nSOME CHECKS FAILED - PLEASE REVIEW")

print("\nSUBMISSION PACKAGE COUNTS:")
print(f"  ├── logs/ ({len(list(Path('results/logs').glob('*')))} files)")
print(f"  ├── metrics/ ({len(list(Path('results/metrics').glob('*')))} files)")
print(f"  ├── histories/ ({len(list(Path('results/histories').glob('*')))} files)")
print(f"  ├── tables/ ({len(list(Path('results/tables').glob('*')))} files)")
print(f"  ├── plots/ ({len(list(Path('results/plots').glob('*')))} files)")
print(f"  └── shap/ ({len(list(Path('results/shap').glob('*')))} files)")


PRE-SUBMISSION VERIFICATION CHECKLIST
✓ Experiments Complete: PASS
✓ Comparison Tables: PASS
✓ Training Curves: PASS
❌ Shap Plots: FAIL
✓ Logs Generated: PASS
✓ Csvs Exported: PASS

SOME CHECKS FAILED - PLEASE REVIEW

SUBMISSION PACKAGE COUNTS:
  ├── logs/ (122 files)
  ├── metrics/ (2 files)
  ├── histories/ (12 files)
  ├── tables/ (2 files)
  ├── plots/ (12 files)
  └── shap/ (0 files)


In [28]:
shutil.make_archive('/kaggle/working/output', 'zip', '/kaggle/working')

'/kaggle/working/output.zip'